# One audit per ontology

For every ontology, this notebook writes a short audit in `audits/<ACRONYM>.md`: its scores, the O'FAIRe
questions where it loses points, the cheatsheet attributes that are empty, and who can fill them in.
It also writes `audits/index.md` (all ontologies, best first) and `data/actions_<RUN_DATE>.csv`, the base
for a curation strategy.

- **I.** Loading what the previous notebooks produced
- **II.** Empty attributes and the points they cost
- **III.** Writing the audits
- **IV.** A first look at the actions

In [1]:
# Imports
import json

import pandas as pd

from config import *

# I. Loading what the previous notebooks produced

### Everything the audits need, in one place

This notebook writes no new analysis of its own: it **combines** the results of the previous notebooks, all
for the same snapshot (`RUN_DATE`).

| Object | What it holds | Made by |
|---|---|---|
| `catalogue` | the ontology record of each entry (acronym, name, administrators, view of, ...) | notebook 01 |
| `submissions` | the latest submission of each entry, with all its metadata | notebook 01 |
| `ofaire` | the full O'FAIRe result of each entry | notebook 01 |
| `scores` | one row per ontology: overall scores and the 61 question scores | notebook 02 |
| `questions` | one row per question: maximum, evaluable, reachable | notebook 02 |
| `guideline` | the machine-readable guideline: 99 attributes, where they are stored, who fills them | notebook 04 |
| `links` | which attributes each O'FAIRe question reads | notebook 04 |

**A detail that matters: the attribute numbers stay text.** The cheatsheet numbers its attributes 1.1, 1.2, ...
1.10. Read as numbers, `1.1` and `1.10` are the same value. If the ids were read as numbers, five pairs would
collapse (1.1 and 1.10, 10.1 and 10.10, ...), and the audits would cite the wrong attribute. `dtype={"id": str}`
tells pandas to read them as text.

**Empty cells become empty text.** In a CSV file, an empty cell comes back as `NaN` ("not a number").
`.fillna("")` replaces it with an empty text, so that simple tests such as `cheatsheet_entry != ""` work.

**Three helpers are prepared for Part II:**

- `reachable`: the 49 questions an EarthPortal ontology can really score on (notebook 02); the 12 others are
  never counted as losses;
- `questions_of`: the reverse of the links table. For each cheatsheet attribute, the set of questions that
  read it. For example, *Version IRI* is read by F1Q4; *License* by I3Q3 and R1.1Q1; *URI*, the most used, by
  three questions. 94 of the 99 attributes are read by at least one question;
- `explanation`: O'FAIRe's own sentence for each (ontology, question), such as *"The ontology is registered
  in 1 ontology repository."* The audits quote it word for word, so that authors read O'FAIRe's reason, not
  ours.

In [2]:
# We load everything the previous notebooks produced for the same snapshot (RUN_DATE)
# From notebook 01, the raw answers of the API and of O'FAIRe:
catalogue = {o["acronym"]: o for o in json.load(open(RAW / "catalogue.json", encoding="utf-8"))}   # acronym -> ontology record
submissions = json.load(open(RAW / "submissions.json", encoding="utf-8"))                         # acronym -> latest submission
ofaire = json.load(open(RAW / "ofaire.json", encoding="utf-8"))                                   # acronym -> O'FAIRe result
# From notebook 02, the scores (one row per ontology) and the questions (one row per question):
scores = pd.read_csv(DATA / f"ofaire_scores_{RUN_DATE}.csv").set_index("acronym")
questions = pd.read_csv(DATA / f"ofaire_questions_{RUN_DATE}.csv").set_index("question")
# From notebook 04, the machine-readable guideline and the links between questions and attributes.
# The id must stay text: read as numbers, "10.1" and "10.10" would both become 10.1
guideline = pd.read_csv(MAPPING / "guideline_profile.csv", dtype={"id": str}).fillna("")   # empty cells: "" instead of NaN
links = pd.read_csv(MAPPING / "ofaire_vs_guideline.csv").fillna("")

# The 49 questions an EarthPortal ontology can really score on (see notebook 02)
reachable = questions[questions.evaluable & questions.reachable].index.tolist()
# For each cheatsheet attribute, the set of questions that read it, e.g. "Version IRI" -> {"F1Q4"}
questions_of = links[links.cheatsheet_entry != ""].groupby("cheatsheet_entry").question.apply(set).to_dict()

# We keep O'FAIRe's own explanation of each score, to quote it in the audits
explanation = {}            # (acronym, question) -> sentence, e.g. ("OSO", "F4Q2") -> "The ontology is registered in 1 ..."
for acronym, result in ofaire.items():
    for principle in ("Findable", "Accessible", "Interoperable", "Reusable"):
        for block in result[principle].values():
            if isinstance(block, dict):                             # the sub-principles; the numbers are skipped
                for qid, q in block.get("results", {}).items():
                    explanation[(acronym, qid)] = (q.get("explanation") or "").strip()
print(f"{len(scores)} ontologies, {len(reachable)} reachable questions, {len(guideline)} cheatsheet attributes")

92 ontologies, 49 reachable questions, 99 cheatsheet attributes


# II. Empty attributes and the points they cost

An attribute is **empty** when its value is null, `""`, `[]` or `{}`; an agent or a contact whose record
has nothing but its identifier counts as empty too. When a question loses *L* points and reads *n* empty
attributes, each of them receives a share of *L / n*. This share is a way to rank the actions, not a
promise of points, because O'FAIRe has its own rule for each question. For example, I1Q5 needs
*Has format* and *Is format of* together, one attribute is enough for I2Q6, and F1Q2 gives its 11 points
only to a valid DOI.

Each empty attribute gets a **priority**: 1 = Mandatory; 2 = Recommended and costing O'FAIRe points;
3 = Recommended only; 4 = Optional but costing O'FAIRe points. Optional attributes that cost nothing are
not actions.

### How the actions of one ontology are computed

The loop takes the ontologies one by one (92 in the snapshot of 8 October) and follows four steps.

**1. The points it loses.** For each of the 49 reachable questions: lost points = maximum − score. Questions
with a full score are left out, and so are the questions without a real answer (empty scores, see notebook 02).

**2. Its empty attributes.** For each of the 99 attributes of the guideline, we read the value where the portal
stores it (the record and the API key found in notebook 04) and decide whether it is **filled**:

| Value | Filled? |
|---|---|
| nothing (`null`), `""`, a text of spaces only | no |
| `[]`, or a list whose elements are all empty | no |
| a record (an agent, a contact) whose fields are all empty, its own identifier and type aside | no |
| `true` or `false`, any number (even 0), any text | yes |

Two attributes are only asked when they make sense: *This ontology is a view of* is asked only of views, and
*Was invalidated by* only of retired ontologies (being invalidated means being withdrawn).

**3. The share of the lost points.** An empty attribute is linked to the questions that read it (notebook 04).
When a question loses points and reads several empty attributes, its lost points are **divided equally**
between them. An attribute read by several losing questions adds up its shares.

> **Example 1, ENVTHES, question R1.2Q1** (actors of the development): score 3 out of 8, so 5 points lost.
> R1.2Q1 reads three empty attributes of ENVTHES: *Contributor*, *Curator* and *Translator*. Each one gets
> 5 / 3 = **1.7** points.
>
> **Example 2, OSO, question I1Q5** (other formats): score 0 out of 4. Of the attributes it reads, only
> *Is format of* is empty, so it gets the whole **4** points.

The share is a way to **rank** the actions, not a promise: O'FAIRe has its own rule for each question (I1Q5,
for instance, needs *Has format* and *Is format of* together). A question that loses points without reading
any empty attribute (a server setting, the number of catalogues) produces no action here: filling a field would
not change it.

**4. The priority.**

| Priority | Attribute | Points at stake | Actions on 8 October |
|---|---|---|---|
| **1** | Mandatory | any | 224 |
| **2** | Recommended | yes | 2,812 |
| **3** | Recommended | no | 1,517 |
| **4** | Optional | yes | 999 |
| none | Optional | no | (not an action) |

Every action keeps who can take it (author, curator or portal), the attribute, the questions concerned and the
share. They are all saved in `data/actions_<RUN_DATE>.csv`: **5,552 actions** on 8 October (3,895 for artefacts,
1,288 for views, 369 for metadata-only records). This file is the base for a curation strategy; Part III turns
it into one readable page per ontology.

In [3]:
def filled(value):
    """We return True when a metadata value really says something, False when it is empty."""
    if value is None:                                   # no value at all
        return False
    if isinstance(value, bool):                         # True and False are both real answers
        return True
    if isinstance(value, str):                          # a text: empty or made of spaces only counts as empty
        return value.strip() != ""
    if isinstance(value, list):                         # a list: filled if at least one element is filled
        return any(filled(v) for v in value)
    if isinstance(value, dict):                         # a record (an agent, a contact): filled if one of its
        return any(filled(v) for k, v in value.items()  # fields says something, its own identifier and type aside
                   if k not in ("id", "@id", "@type", "@context"))
    return True                                         # a number (even 0) or any other value: filled


def value_of(acronym, attribute):
    """We return the value of one cheatsheet attribute for one ontology, read where the portal stores it."""
    record = catalogue[acronym] if attribute.portal_object == "ontology" else submissions.get(acronym, {})
    return record.get(attribute.api_key)                # None if the API did not return this key


actions = []                # one row per (ontology, empty attribute worth an action)
lost_points = {}            # acronym -> {question: points lost}, kept for the audits of Part III
for acronym, s in scores.iterrows():                    # one ontology at a time; s is its row of scores
    kind = s.kind
    # 1. The points lost on each reachable question: maximum minus score (empty scores and full scores skipped)
    lost = {q: questions.loc[q, "max_credits"] - s[q] for q in reachable
            if pd.notna(s[q]) and s[q] < questions.loc[q, "max_credits"]}
    lost_points[acronym] = lost
    # 2. The cheatsheet attributes that are empty for this ontology
    empty = guideline[[not filled(value_of(acronym, a)) for a in guideline.itertuples()]]
    if kind != "view":                                  # only a view can be "a view of" something
        empty = empty[empty.entry != "This ontology is a view of"]
    if submissions.get(acronym, {}).get("status") != "retired":   # only a retired ontology "was invalidated"
        empty = empty[empty.entry != "Was invalidated by"]
    # 3. For each question that lost points: how many of the empty attributes does it read?
    n_empty = {q: sum(q in questions_of.get(e, set()) for e in empty.entry) for q in lost}
    for a in empty.itertuples():                        # each empty attribute of this ontology
        qs = sorted(q for q in questions_of.get(a.entry, set()) if q in lost)   # the losing questions that read it
        share = round(sum(lost[q] / n_empty[q] for q in qs), 1)                 # its share of their lost points
        # 4. The priority: 1 Mandatory; 2 Recommended and costing points; 3 Recommended only;
        #    4 Optional and costing points; None (no action) for an Optional attribute that costs nothing
        priority = {"Mandatory": 1, "Recommended": 2 if qs else 3}.get(a.usage, 4 if qs else None)
        if priority:
            actions.append({"acronym": acronym, "kind": kind, "priority": priority, "who_fixes": a.who_fixes,
                            "cheatsheet_id": a.id, "entry": a.entry, "usage": a.usage,
                            "portal_attribute": a.portal_attribute, "questions": " ".join(qs),
                            "points_share": share})
actions = pd.DataFrame(actions)                         # the base for a curation strategy
actions.to_csv(DATA / f"actions_{RUN_DATE}.csv", index=False)
print(f"{len(actions)} actions saved to {(DATA / f'actions_{RUN_DATE}.csv').relative_to(REPO)}")
actions.head()

5552 actions saved to earthportal_sa_audit/data/actions_2026-10-16.csv


,acronym,kind,priority,who_fixes,cheatsheet_id,entry,usage,portal_attribute,questions,points_share
0,OSO,artefact,2,author,6.2,Is format of,Recommended,dct:isFormatOf,I1Q5,4.0
1,OSO,artefact,3,author,8.6,Award,Recommended,schema:award,,0.0
2,OSO,artefact,3,author,10.4,Similar to,Recommended,door:similarTo,,0.0
3,OSO,artefact,3,author,10.10,Disparate modelling with,Recommended,door:hasDisparateModelling,,0.0
4,OSO,artefact,3,author,10.15,Used by,Recommended,voaf:usedBy,,0.0


# III. Writing the audits

### One page per ontology, and an index

This cell turns the actions of Part II into pages that a person can read: one Markdown file per ontology in
`audits/<ACRONYM>.md`, and `audits/index.md`, which lists them all. Markdown is plain text with a few marks
(`#` for a title, `|` for a table, `**` for bold); GitHub displays it as a formatted page, so an author can
read their audit directly in the repository, without installing anything.

**What one audit page contains** (example: ENVTHES on 8 October):

1. **The title and the context**: acronym, name, kind (artefact, view or metadata-only), snapshot date, and a
   link to the page of the ontology on EarthPortal.
2. **The scores**: O'FAIRe's own score (57 / 100), the points (273 / 478), the reachable score (70.7 % of 386)
   and the rank among the artefacts (23 of 68). Views and metadata-only records are not ranked. When two
   artefacts have the same score, they share the best rank: NVS_P06 and NVS_P07, both at 74.5 %, are both 4th.
3. **Where O'FAIRe points are lost**: one row per question, the largest loss first, with O'FAIRe's own
   sentence, quoted as it is (for ENVTHES, the first row is R1.3Q1, 10 points lost: *"The ontology does not
   provide information about projects or endorsing organizations."*).
4. **The suggested actions** of priorities 1 and 2, grouped by who can take them (author, curator, portal),
   each with the attribute, its cheatsheet number, the property where the portal stores it and, when it costs
   points, its share and the questions concerned:

```
   - [P2] Fill **Is format of** (cheatsheet 6.2, Recommended, `dct:isFormatOf`); share of the O'FAIRe points lost: 4 (I1Q5)
```

5. **Later**: the attributes of priorities 3 and 4, in one sentence, so that the page stays short.

**The index** has three tables (artefacts, views, records without content), each sorted by reachable score,
with the number of P1 and P2 actions of each ontology. Each acronym is a link to its audit page.

**Two details that keep the pages correct:**

- A name or an O'FAIRe sentence could contain a `|`, which Markdown reads as a column separator, or a line
  break, which would end the table row. The small function `md` replaces them before any text enters a table.
- Numbers are written with the format `:g`, which drops useless zeros: `9` and not `9.0`, `364.36` as it is.

The pages are written again at every run. Since `audits/` is in the repository, git shows, snapshot after
snapshot, exactly what changed in the audit of each ontology.

In [4]:
# We remove the pages of the previous run first, so that an ontology that left the catalogue
# does not keep an outdated audit in the folder
for old_page in AUDITS.glob("*.md"):
    old_page.unlink()

In [5]:
def md(text):
    """We make a text safe inside a Markdown table: no column separator, no line break."""
    return str(text).replace("|", "\\|").replace("\n", " ").strip()

# The rank of each artefact by reachable score; ties share the best rank (two artefacts at 74.5 % are both 4th)
rank = scores[scores.kind == "artefact"].reachable_score_pct.rank(ascending=False, method="min")
n_artefacts = int((scores.kind == "artefact").sum())
# The three groups of actions, with the title they get in an audit
WHO = {"author": "By the author (in the file or the submission form)",
       "curator": "By the curator (EarthPortal team)",
       "portal": "By the portal (generated values: check that processing succeeded)"}

for acronym, s in scores.iterrows():                    # one audit page per ontology
    # Its catalogue record, and its actions: most urgent first, then the largest share first
    onto, mine = catalogue[acronym], actions[actions.acronym == acronym].sort_values(["priority", "points_share"], ascending=[True, False])
    # The page is built as a list of lines, joined at the end. First: the title and the score table
    lines = [f"# {acronym}: {md(onto['name'])}", "",
             f"Kind: **{s.kind}** · Snapshot: {RUN_DATE} · Portal page: https://earthportal.eu/ontologies/{acronym}", "",
             "| O'FAIRe score | Points | Reachable score | Rank |", "|---|---|---|---|",
             f"| {s.normalized_score} / 100 | {s.total_points:g} / 478 | {s.reachable_score_pct} % of {s.reachable_max} | "
             + (f"{int(rank[acronym])} of {n_artefacts} artefacts |" if s.kind == "artefact" else f"not ranked ({s.kind}) |"),
             "", "## Where O'FAIRe points are lost", ""]
    # The lost points, largest first, with O'FAIRe's own sentence for each question
    lost = sorted(lost_points[acronym].items(), key=lambda item: -item[1])
    if lost:
        lines += ["| Question | Points | Lost | O'FAIRe says |", "|---|---|---|---|"]
        lines += [f"| {q} | {s[q]:g} / {questions.loc[q, 'max_credits']:g} | {l:g} | {md(explanation.get((acronym, q), ''))} |" for q, l in lost]
    else:
        lines.append("No points lost on the reachable questions.")
    # The actions of priorities 1 and 2, grouped by who can take them
    lines += ["", "## Suggested actions", "", "Priority 1: Mandatory. Priority 2: Recommended and costing O'FAIRe points. The share is the points lost "
              "on the questions in brackets, divided among the empty attributes these questions read.", ""]
    for who, title in WHO.items():
        todo = mine[(mine.who_fixes == who) & (mine.priority <= 2)]
        if len(todo):                                   # a group without action gets no heading
            lines += [f"### {title}", ""]
            lines += [f"- [P{a.priority}] Fill **{a.entry}** (cheatsheet {a.cheatsheet_id}, {a.usage}, `{a.portal_attribute}`)"
                      + (f"; share of the O'FAIRe points lost: {a.points_share:g} ({a.questions})" if a.questions else "")
                      for a in todo.itertuples()]
            lines.append("")
    # Priorities 3 and 4 in one sentence, so that the page stays short
    later = mine[mine.priority >= 3].entry.tolist()
    if later:
        lines.append(f"Later (priorities 3 and 4): {', '.join(later)}.")
    (AUDITS / f"{acronym}.md").write_text("\n".join(lines) + "\n", encoding="utf-8")   # audits/<ACRONYM>.md

# The index: every ontology, best reachable score first, with its number of P1 and P2 actions
p1 = actions[actions.priority == 1].groupby("acronym").size()      # acronym -> number of P1 actions
p2 = actions[actions.priority == 2].groupby("acronym").size()      # acronym -> number of P2 actions
index = ["# EarthPortal audits", "", f"Snapshot of {RUN_DATE}. *P1*: empty Mandatory attributes; "
         "*P2*: empty Recommended attributes that cost O'FAIRe points.", ""]
for kind, title in (("artefact", "Artefacts"), ("view", "Views"), ("metadata-only", "Records without content")):
    part = scores[scores.kind == kind].sort_values("reachable_score_pct", ascending=False)
    index += [f"## {title} ({len(part)})", "", "| Acronym | Name | O'FAIRe | Reachable | P1 | P2 |", "|---|---|---|---|---|---|"]
    index += [f"| [{a}]({a}.md) | {md(r['name'])} | {r.normalized_score} | {r.reachable_score_pct} % | {p1.get(a, 0)} | {p2.get(a, 0)} |"
              for a, r in part.iterrows()]               # [OSO](OSO.md) is a link to the audit page next to the index
    index.append("")
(AUDITS / "index.md").write_text("\n".join(index), encoding="utf-8")
print(f"{len(scores)} audits and the index written to {AUDITS.relative_to(REPO)}/")

92 audits and the index written to earthportal_sa_audit/audits/


In [6]:
# We look at one audit, the best artefact
print((AUDITS / f"{scores[scores.kind == 'artefact'].reachable_score_pct.idxmax()}.md").read_text(encoding="utf-8"))

# OSO: Observatories of the Seas Ontology

Kind: **artefact** · Snapshot: 2026-10-16 · Portal page: https://earthportal.eu/ontologies/OSO

| O'FAIRe score | Points | Reachable score | Rank |
|---|---|---|---|
| 76 / 100 | 364.36 / 478 | 94.4 % of 386 | 1 of 68 artefacts |

## Where O'FAIRe points are lost

| Question | Points | Lost | O'FAIRe says |
|---|---|---|---|
| F4Q2 | 2 / 10 | 8 | The ontology is registered in 1 ontology repository. |
| A1Q3 | 18 / 24 | 6 | Ontology and its metadata support 6 negotiation formats. (Ontology accept the following formats: [text/html], Ontology metadata accept the following formats: [application/json, application/xml, text/html, text/plain, application/rdf+xml]) |
| I1Q5 | 0 / 4 | 4 | No information is provided about the availability of the ontology in other formats. |
| R1.2Q2 | 4 / 6 | 2 | The ontology provides 2 properties to inform on its general provenance. |
| I2Q7 | 3.36 / 5 | 1.64 | The ontology declares reusing standards metadata vocabular

# IV. A first look at the actions

### What the actions tell us, all ontologies together

The file `data/actions_<RUN_DATE>.csv` can be explored in a spreadsheet, but two questions come first:
**who** has the most to do, and **which Mandatory attributes** are most often empty. We look only at the
artefacts (views and metadata-only records follow other rules) and at priorities 1 and 2.

**Who has the most to do** (8 October, 68 artefacts):

| Who | P1 (Mandatory) | P2 (Recommended, costing points) | All |
|---|---|---|---|
| author | 89 | 1,819 | 1,908 |
| curator | 20 | 61 | 81 |
| portal | 37 | 0 | 37 |
| **all** | **146** | **1,880** | **2,026** |

Most of the work falls on the **authors**: an audit page is mainly a message to them. The **curator** rows
(81) are work for the EarthPortal team itself, and the **portal** rows (37) point to values the portal should
have generated; they are a technical check, not a request to anyone.

**The Mandatory attributes most often empty:**

| Attribute | Artefacts | Who | Remark |
|---|---|---|---|
| Modification date | 27 | author | |
| Location | 24 | author | the pull URL is empty **by design** when a file is uploaded rather than pulled: should it stay Mandatory? |
| Creator | 23 | author | includes agent records with no content |
| Categories | 20 | curator | |
| License | 15 | author | |
| CSV dump | 12 | portal | the CSV export was not produced |
| Download URL | 9 | portal | |
| URI lookup endpoint | 9 | portal | |
| Free-text search endpoint | 7 | portal | |

The split between author, curator and portal is our proposal (notebook 04). It should be validated with the
EarthPortal team before any author is contacted.

In [7]:
# We look at one audit, the best artefact: the one with the highest reachable score (OSO on 8 October)
best = scores[scores.kind == "artefact"].reachable_score_pct.idxmax()      # the acronym of the best artefact
print((AUDITS / f"{best}.md").read_text(encoding="utf-8"))                # its page, as written by Part III

# OSO: Observatories of the Seas Ontology

Kind: **artefact** · Snapshot: 2026-10-16 · Portal page: https://earthportal.eu/ontologies/OSO

| O'FAIRe score | Points | Reachable score | Rank |
|---|---|---|---|
| 76 / 100 | 364.36 / 478 | 94.4 % of 386 | 1 of 68 artefacts |

## Where O'FAIRe points are lost

| Question | Points | Lost | O'FAIRe says |
|---|---|---|---|
| F4Q2 | 2 / 10 | 8 | The ontology is registered in 1 ontology repository. |
| A1Q3 | 18 / 24 | 6 | Ontology and its metadata support 6 negotiation formats. (Ontology accept the following formats: [text/html], Ontology metadata accept the following formats: [application/json, application/xml, text/html, text/plain, application/rdf+xml]) |
| I1Q5 | 0 / 4 | 4 | No information is provided about the availability of the ontology in other formats. |
| R1.2Q2 | 4 / 6 | 2 | The ontology provides 2 properties to inform on its general provenance. |
| I2Q7 | 3.36 / 5 | 1.64 | The ontology declares reusing standards metadata vocabular

In [8]:
# Artefacts only, priorities 1 and 2: who has the most to do?
top = actions[(actions.kind == "artefact") & (actions.priority <= 2)]
pd.crosstab(top.who_fixes, top.priority, margins=True)    # rows: who; columns: priority; margins: the totals "All"

priority,1,2,All
who_fixes,,,
author,89,1819,1908
curator,20,61,81
portal,37,0,37
All,146,1880,2026


In [9]:
# The Mandatory attributes most often empty, among the artefacts
actions[(actions.kind == "artefact") & (actions.priority == 1)].entry.value_counts()

entry
Modification date            27
Location                     24
Creator                      23
Categories                   20
License                      15
CSV dump                     12
Download URL                  9
URI lookup Endpoint           9
Free-text search endpoint     7
Name: count, dtype: int64

**Conclusion**: most of the work falls on the authors, and a small part on the EarthPortal team. The
`who_fixes` column is a proposal to discuss with the team before contacting anyone.